# 04 Word Embeddings

## Housekeeping (no interaction required)

In [2]:
%pip install gensim

/home/bode-wsl/projects/summerschool26/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import random
import time
from pathlib import Path

import gensim
import pandas as pd
from tqdm.notebook import tqdm

tqdm.pandas()

In [2]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBSummerSchool26/data') if IN_COLAB else Path('../data')

In [3]:
def confirm(question: str = "Do you want to execute this cell?"):
    while True:
        response = input(f"{question} (y/n): ").lower()
        if response in ["y", "yes"]:
            return True
        elif response in ["n", "no"]:
            return False
        else:
            print("Please enter 'y' or 'n'.")

## Setup (Interaction required)

In [4]:
### ⬇️⬇️⬇️ 💽 Adjust here if you want to continue with your own query
CORPUS_NAME = "armensteuer_and_similars"
LOAD_OWN_DATA = True
### ⬆️⬆️⬆️

💽 You only need to run the cell below if you want to work with your own query.

*Once prompted, give all demanded permissions*

In [5]:
if IN_COLAB and confirm("Do you want to mount your Google Drive?"):
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

## Load the data

### <img src="https://cdn.simpleicons.org/googledrive" alt="💾" width=16> Load your own data from Google Drive

### <img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=16> Load your own data from Google Drive

In [6]:
if LOAD_OWN_DATA:
    RAWDATA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.parquet" # Use data from filtering module
    raw_df = pd.read_parquet(RAWDATA_PATH)

    LEMMA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.lemmas.parquet"
    lemma_df = pd.read_parquet(LEMMA_PATH)

    TOKENS_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.tokens.parquet"
    wordform_df = pd.read_parquet(TOKENS_PATH)

    raw_df = raw_df.join(lemma_df)
    raw_df = raw_df.join(wordform_df)

### <img src="https://cdn.simpleicons.org/github" alt="🏫" width=16> Load from Github

### <img src="https://www.zb.uzh.ch/themes/zb/assets/images/favicon-192.png" alt="💾" width=16> Load from example

In [13]:
if not LOAD_OWN_DATA:
    # RAWDATA_ORIGIN_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.csv" 
    # raw_df = pd.read_csv(RAWDATA_ORIGIN_URL, index_col="id")

    RAWDATA_ORIGIN_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.parquet"
    print(f"Loading raw data ...", end="\r")
    raw_df = pd.read_parquet(RAWDATA_ORIGIN_URL)

    LEMMA_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.lemmas.parquet"
    print(f"Loading lemma data ...", end="\r")
    lemma_df = pd.read_parquet(LEMMA_URL)

    TOKENS_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.tokens.parquet"
    print(f"Loading token data ...", end="\r")
    wordform_df = pd.read_parquet(TOKENS_URL)

    raw_df = raw_df.join(lemma_df)
    raw_df = raw_df.join(wordform_df)

    print(f"Data loaded successfully! {len(raw_df)} rows.")

Data loaded successfully! 3175 rows.


## Parsing

In [7]:
import numpy as np

def convert_iterable_to_list(obj):
    """Recursively convert nested ndarrays to nested lists."""
    if isinstance(obj, np.ndarray):
        return [convert_iterable_to_list(item) for item in obj]
    else:
        return obj

raw_df['tokens'] = raw_df['tokens'].progress_apply(convert_iterable_to_list)
raw_df['lemmas'] = raw_df['lemmas'].progress_apply(convert_iterable_to_list)

  0%|          | 0/3175 [00:00<?, ?it/s]

  0%|          | 0/3175 [00:00<?, ?it/s]

# Word2Vec

In [15]:
from gensim.models import Word2Vec

In [22]:
# Memory-efficient way to iterate over tokenized sentences
class SentenceIterator:
    def __init__(self, df, column):
        self.df = df
        self.column = column

    def __iter__(self):
        for document in tqdm(self.df[self.column]):
            for sentence in document:
                yield sentence

sentences = SentenceIterator(raw_df, 'tokens')

In [23]:
model = Word2Vec(sentences=sentences, vector_size=300, window=5, min_count=5, workers=4)

  0%|          | 0/3175 [00:00<?, ?it/s]

  0%|          | 0/3175 [00:00<?, ?it/s]

  0%|          | 0/3175 [00:00<?, ?it/s]

  0%|          | 0/3175 [00:00<?, ?it/s]

  0%|          | 0/3175 [00:00<?, ?it/s]

  0%|          | 0/3175 [00:00<?, ?it/s]

In [8]:
from gensim.models import KeyedVectors
if LOAD_OWN_DATA:
    model = KeyedVectors.load_word2vec_format(str(DATA_DIR / f"{CORPUS_NAME}.word_vectors"), binary=False)

In [ ]:
model.mo

In [10]:
wv = model.wv

AttributeError: 'KeyedVectors' object has no attribute 'wv'

In [ ]:
wv.save_word2vec_format(DATA_DIR / f"{CORPUS_NAME}.word_vectors", binary=False)

In [19]:
model.most_similar("Armensteuer")

[('Steuer', 0.868951678276062),
 ('Gemeindesteuer', 0.8420543670654297),
 ('Staatssteuer', 0.8001058101654053),
 ('Gemeindesteuern', 0.7965085506439209),
 ('Armensteuern', 0.789980947971344),
 ('Taxe', 0.739831805229187),
 ('Kirchensteuer', 0.7390082478523254),
 ('Ansatz', 0.7269775867462158),
 ('Promille', 0.7229862213134766),
 ('Prämie', 0.71816486120224)]

In [ ]:
QUERY = "Lumpen"

hits = raw_df[raw_df["content"].str.contains(QUERY)]
hit = hits[["content"]][:1]
hit.iloc[0]["content"]

'« tacke « Gebitt mit etwa 5000 bi « sooo « in. wohnern erlauft wurde, eines Stücke «, welches nöthig Genf hatte, um feine einzelnen zerstreut in Savoyen liegenden Gebietstheil « mit « inander zu verbinden." In ben, V »« l. dagegen, Nachr." schreibt der Einsender im Journal " weiter, stand vor einiger geit zu lesen: » Die Möglichkeit für die Schweiz, sich in Genf und im Wallis zu behaupten, hängt wesentlich oon der Besetzung deS Chablais und Faucigny ab. so daß bie militärische Vesih. nahme dieser beiden Provinzen durch Fran!» 7nch eine direkte Drohung gegenüber Oberitalien ist und das ist heute richtig ebenso als im Iahre 1615.- Dieser zivil « und militärische Besitz ist Frankreich durch den König von Italien übertragen » orden als Gegenwert!) der von ben französischen Truppen eroberten und an Viktor Emanuel ab » getretenen Lombard «, König welcher in ber Ab « tretung Savoyen « an Frankreich wie es scheint leine direkte Bedrohung Italiens gesehen hat. Die Neutralisation Savoyens hat ü

In [ ]:
model.most_similar_cosmul(positive=["Armut", "unverschuldet"], negative=["verschuldet"])[:5]

[('Behinderter', 1.015238881111145),
 ('Gebrechliche', 0.9977672100067139),
 ('Landwirthschaft', 0.9969664216041565),
 ('Unverletzlichkeit', 0.99241042137146),
 ('notleidender', 0.9878964424133301)]

In [18]:
model.most_similar_cosmul(positive=["Armensteuer", "gerecht"], negative=["ungerecht"])[:5]

[('Steuer', 0.9902782440185547),
 ('steuern', 0.9862464070320129),
 ('Last', 0.9681793451309204),
 ('Deckung', 0.9631245732307434),
 ('Gemeindesteuer', 0.9406282305717468)]

In [ ]:
import csv

def wvec2tsv(infile, outf_vecs, outf_meta):
    with open(infile, "r", encoding='utf-8') as vec_file:
        with open(outf_vecs, "w", encoding='utf-8') as vec_tsv:
            with open(outf_meta, "w", encoding='utf-8') as meta_tsv:
                tsv_writer1 = csv.writer(vec_tsv, delimiter='\t')
                tsv_writer2 = csv.writer(meta_tsv, delimiter='\t')
                # erste Linie überspringen, sie ist ohne relevanten Inhalt
                line = vec_file.readline()
                line = vec_file.readline()
                currvec = line.strip().split(' ')
                counter = 0
                pbar = tqdm(total=model.vectors.shape[0], desc="Converting word vectors to TSV format")
                while line:
                    counter += 1
                    if currvec[0] == "":
                        break
                    tsv_writer2.writerow([currvec[0]])
                    currvec.pop(0)
                    tsv_writer1.writerow(currvec)
                    currvec = vec_file.readline().strip().split(' ')
                    pbar.update(1)
                pbar.close()
                print("Done!")

wvec2tsv(str(DATA_DIR / f"{CORPUS_NAME}.word_vectors"), str(DATA_DIR / f"{CORPUS_NAME}.vecs.tsv"), str(DATA_DIR / f"{CORPUS_NAME}.meta.tsv"))

Converting word vectors to TSV format:   0%|          | 0/59372 [00:00<?, ?it/s]

Done!
